# Look at mapping breadth and depth of 100+ metagenomes to singleclust genes - highcov version 

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [3]:
DIR='../outputs.cds/singleclust.highcov/bam'
template = '../outputs.cds/singleclust.highcov/bam/{metag}.x.{species}.depth.txt'

def read_depth_txt(metag, species, *, exclude_ends=75):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo')).select(['gene', 'pos', 'cov'])

    sum_df = df.group_by('gene').all().with_columns(
        # select slice [75:-75]
        (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species")),
        # summarize: average depth across contig,
        (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
        # average depth across covered bases,
        (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
        # fraction of bases covered
        (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
    ).select(["metag", "species", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    return sum_df

read_depth_txt('SRR8960963', 's__Cryptobacteroides sp900546925')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""NMJBBHDE_00454""",771,771,1.0,25.175097,25.175097
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""LELOEEPG_00175""",1047,879,0.839542,12.006686,14.301479
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""MFHJPMDA_01033""",1137,998,0.877748,2.664908,3.036072
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_02574""",1101,0,0.0,0.0,NaN
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_00373""",1644,1639,0.996959,25.844891,25.923734
…,…,…,…,…,…,…,…
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""EHLNOCPD_01770""",765,499,0.652288,13.566013,20.797595
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""NLDKNCOF_00688""",1308,1304,0.996942,17.939602,17.994632
"""SRR8960963""","""s__Cryptobacteroides sp9005469…","""HLFEEHJE_00119""",177,177,1.0,15.350282,15.350282


In [4]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth_txt(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1753
100 of 1753
200 of 1753
300 of 1753
400 of 1753
500 of 1753
600 of 1753
700 of 1753
800 of 1753
900 of 1753
1000 of 1753
1100 of 1753
1200 of 1753
1300 of 1753
1400 of 1753
1500 of 1753
1600 of 1753
1700 of 1753
read 1753 depth files.


In [5]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""SRR17241520""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,48573.501425,48573.501425
"""SRR17241677""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,44080.032764,44080.032764
"""SRR17241632""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,30802.037037,30802.037037
"""SRR17241654""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,27947.760684,27947.760684
"""SRR17241672""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,25236.007123,25236.007123
…,…,…,…,…,…,…,…
"""SRR11185261""","""s__Sodaliphilus sp004557565""","""EOMKCHNH_02137""",1290,2,0.00155,0.00155,1.0
"""SRR16235689""","""s__Prevotella sp002251295""","""JKBOFALP_01720""",2001,3,0.001499,0.001499,1.0
"""SRR17241643""","""s__Prevotella sp000434975""","""KJGAGFBO_01044""",2091,3,0.001435,0.001435,1.0


## Export mapping abundance data to a CSV, after merging with `species-genes.csv`

In [6]:
species_genes_df = (pl.read_csv('../outputs.cds/singleclust/species-genes.csv')
    .filter(pl.col("good") == 1)
    .with_columns(pl.col('gene_name').alias('gene'))
    .select(["anchor", "gene", "species", "description"])
)

species_genes_df

anchor,gene,species,description
i64,str,str,str
1,"""CNENGHLA_01260""","""s__Phascolarctobacterium_A suc…","""BLAST match to hydrogenase lar…"
0,"""EHOAPHDI_01174""","""s__Phascolarctobacterium_A suc…","""BLAST match to protein phospha…"
0,"""CNENGHLA_00658""","""s__Phascolarctobacterium_A suc…","""BLAST match to 4-hydroxy-3-met…"
0,"""IFIBFMPA_00800""","""s__Phascolarctobacterium_A suc…","""BLAST match to 2-isopropylmal…"
0,"""BBOFCOCJ_01349""","""s__Lactobacillus amylovorus""","""BLAST match to peptidase T [La…"
…,…,…,…
1,"""JBPBJODD_00501""","""s__Cryptobacteroides sp0004349…","""hypothetical protein [Bacteroi…"
0,"""KHMCBGLI_01458""","""s__Cryptobacteroides sp0004349…","""putative uncharacterized prote…"
0,"""FCHBMNJF_01297""","""s__Cryptobacteroides sp0004349…","""ATP-binding protein [Bacteroid…"


In [7]:
merge_df = depth_df.join(species_genes_df, on=["species", "gene"], how='inner')
merge_df

metag,species,gene,len,hits,breadth,depth_all,depth_cov,anchor,description
str,str,str,u32,u32,f64,f64,f64,i64,str
"""SRR8960206""","""s__Floccifex porci""","""IIAENJBH_00799""",1848,793,0.429113,0.511905,1.192938,1,"""anaerobic ribonucleoside-triph…"
"""SRR8960441""","""s__Cryptobacteroides sp0004349…","""JBPBJODD_01157""",1074,1024,0.953445,4.93203,5.172852,0,""" relA/SpoT family protein [Bac…"
"""SRR8960441""","""s__Cryptobacteroides sp0004349…","""KHMCBGLI_01458""",1734,1475,0.850634,3.291811,3.869831,0,"""putative uncharacterized prote…"
"""SRR8960441""","""s__Cryptobacteroides sp0004349…","""JBPBJODD_00501""",2034,1896,0.932153,5.193215,5.571203,1,"""hypothetical protein [Bacteroi…"
"""SRR8960441""","""s__Cryptobacteroides sp0004349…","""FCHBMNJF_01297""",3525,3525,1.0,5.731915,5.731915,0,"""ATP-binding protein [Bacteroid…"
…,…,…,…,…,…,…,…,…,…
"""SRR17241640""","""s__Prevotella sp000434975""","""APOLKBKG_00160""",3177,2137,0.672647,4.85395,7.216191,1,"""efflux RND transporter permeas…"
"""SRR17241640""","""s__Prevotella sp000434975""","""IGEONAAO_00273""",423,260,0.614657,6.219858,10.119231,0,"""hypothetical protein"""
"""SRR18048908""","""s__Prevotella sp000434975""","""APOLKBKG_00160""",3177,3177,1.0,16.341832,16.341832,1,"""efflux RND transporter permeas…"


In [8]:
merge_df.write_csv('../outputs.cds/cds3-genes/mapping-coverage-highcov-metag.csv')

In [9]:
print(len(merge_df))
print(len(species_genes_df))

7131
69


In [10]:
species_genes_df['gene'].n_unique()

69

In [11]:
merge_df['gene'].n_unique()

61